# W5 · Day 1 — Eval Mechanics Playground

**~90 minutes · in-class demo · Jupyter notebook · Track A**

Today's session builds the four eval patterns you'll port to your capstone
in Day 2. We do it on a small, non-capstone dataset first — 6 movie plot
summaries — so the mechanics are the ONLY thing you're thinking about.

Tomorrow (Day 2) you'll build a 20-question golden set for YOUR capstone
corpus and run these same patterns against your own `/ask` endpoint. That's
the milestone (M1). Today is the mental model.

**The four patterns we build:**
1. Rubric-based LLM-as-judge (Section 4) — score answers 1-4 on 3 dimensions
2. Pairwise comparison (Section 5) — 'which is better, A or B?'
3. Position bias — see it happen, then mitigate (Section 6)
4. Critic-Creator loop (Section 7) — systematic prompt improvement

**Cost per full run:** ~$0.05 in OpenAI credits (gpt-4o is the judge —
a bit pricier than gpt-4o-mini, but you need a strong judge).

---

## Section 1 — Setup + corpus

6 short movie plot summaries. Small enough to inline. Diverse enough that
questions can be easy, tricky, or edge cases.

This corpus stays with us all session.

In [1]:
import os
import json
import random
from openai import OpenAI
from pydantic import BaseModel, Field

assert os.environ.get("OPENAI_API_KEY"), "Set OPENAI_API_KEY before running this notebook"

client = OpenAI()

# gpt-4o for the judge — you need a strong model here.
# gpt-4o-mini for the candidate answers — that's what a production system
# would ship. Judge is stronger than judged, always.
JUDGE_MODEL     = "gpt-4o"
CANDIDATE_MODEL = "gpt-4o-mini"

print(f"Judge model:     {JUDGE_MODEL}")
print(f"Candidate model: {CANDIDATE_MODEL}")
print("Setup ok.")

Judge model:     gpt-4o
Candidate model: gpt-4o-mini
Setup ok.


In [2]:
MOVIES = [
    {"id": "matrix", "title": "The Matrix", "plot": (
        "A computer programmer named Neo discovers that the world he lives in "
        "is a simulated reality called the Matrix, created by intelligent "
        "machines to subdue humanity. Recruited by rebel leader Morpheus, Neo "
        "learns to bend the rules of the simulation. He is prophesied to be "
        "'the One' who can end the war between humans and machines."
    )},
    {"id": "inception", "title": "Inception", "plot": (
        "Dom Cobb is a thief who steals corporate secrets by infiltrating "
        "people's dreams. He is offered a chance to have his criminal record "
        "erased if he can perform 'inception' — planting an idea in a target's "
        "mind rather than stealing one. He assembles a team and enters nested "
        "dream levels within dream levels, risking becoming trapped in limbo."
    )},
    {"id": "jurassic_park", "title": "Jurassic Park", "plot": (
        "A wealthy entrepreneur secretly creates a theme park on a remote "
        "island featuring living dinosaurs cloned from ancient DNA. Before "
        "opening to the public, he invites a paleontologist, a paleobotanist, "
        "and a mathematician to endorse the park. During a storm, the "
        "security systems fail and the dinosaurs escape their enclosures."
    )},
    {"id": "toy_story", "title": "Toy Story", "plot": (
        "Woody the cowboy doll is the favorite toy of a boy named Andy — "
        "until Andy receives Buzz Lightyear, a space ranger action figure, "
        "for his birthday. Jealous of Buzz's popularity, Woody accidentally "
        "knocks him out the window. The two rivals must work together to "
        "find their way back to Andy before the family moves house."
    )},
    {"id": "titanic", "title": "Titanic", "plot": (
        "Aboard the RMS Titanic in 1912, a poor artist named Jack Dawson "
        "falls in love with a wealthy young woman, Rose DeWitt Bukater, who "
        "is engaged to an arrogant heir. Their romance unfolds against the "
        "backdrop of the ship's maiden voyage. On the fourth night at sea, "
        "the ship strikes an iceberg and begins to sink."
    )},
    {"id": "godfather", "title": "The Godfather", "plot": (
        "Don Vito Corleone is the aging patriarch of a powerful New York "
        "Mafia family in the 1940s. When rival families propose entering the "
        "narcotics trade and Don Vito refuses, a war erupts. His youngest "
        "son Michael, initially uninvolved in the family business, is drawn "
        "in after his father survives an assassination attempt."
    )},
]

print(f"Loaded {len(MOVIES)} movies.\n")
for m in MOVIES:
    print(f"  {m['id']:15s}  {m['title']:20s}  {len(m['plot'])} chars")

Loaded 6 movies.

  matrix           The Matrix            328 chars
  inception        Inception             340 chars
  jurassic_park    Jurassic Park         325 chars
  toy_story        Toy Story             319 chars
  titanic          Titanic               310 chars
  godfather        The Godfather         318 chars


In [3]:
# Six questions. Two easy, two medium, one edge case, one requiring inference.
QUESTIONS = [
    {"id": "q1", "question": "What is the name of the rebel leader who recruits Neo?",
     "ideal": "Morpheus", "difficulty": "easy"},
    {"id": "q2", "question": "Who is Andy's favorite toy at the start of Toy Story?",
     "ideal": "Woody", "difficulty": "easy"},
    {"id": "q3", "question": "In Inception, what happens if you get trapped in limbo?",
     "ideal": "You risk becoming stuck in the deepest dream level (the plot doesn't fully explain).",
     "difficulty": "medium"},
    {"id": "q4", "question": "Compare how the conflict starts in The Godfather versus Jurassic Park.",
     "ideal": "Godfather: rival families propose narcotics trade, Don refuses, war erupts. "
              "Jurassic Park: storm causes security failure, dinosaurs escape.",
     "difficulty": "medium"},
    {"id": "q5", "question": "What year does the Star Wars trilogy take place in?",
     "ideal": "Not covered — Star Wars is not in the corpus.",
     "difficulty": "edge (out-of-scope)"},
    {"id": "q6", "question": "Which movies from the corpus involve a romantic relationship?",
     "ideal": "Titanic (Jack and Rose).",
     "difficulty": "inference"},
]

print(f"{len(QUESTIONS)} questions loaded.\n")
for q in QUESTIONS:
    print(f"  {q['id']}  ({q['difficulty']:22s}) {q['question']}")

6 questions loaded.

  q1  (easy                  ) What is the name of the rebel leader who recruits Neo?
  q2  (easy                  ) Who is Andy's favorite toy at the start of Toy Story?
  q3  (medium                ) In Inception, what happens if you get trapped in limbo?
  q4  (medium                ) Compare how the conflict starts in The Godfather versus Jurassic Park.
  q5  (edge (out-of-scope)   ) What year does the Star Wars trilogy take place in?
  q6  (inference             ) Which movies from the corpus involve a romantic relationship?


---

## Section 2 — Generate naive answers

Stuff the whole corpus into the prompt (naive — no retrieval, that's W6). Get
an answer to each question. These become our **candidates to grade**.

In [4]:
def answer_naively(question: str, corpus: list, temperature: float = 0.0) -> str:
    """Stuff the whole corpus into the prompt. Ask the question."""
    context = "\n\n".join(f"=== {m['title']} ===\n{m['plot']}" for m in corpus)
    resp = client.chat.completions.create(
        model=CANDIDATE_MODEL,
        temperature=temperature,
        messages=[
            {"role": "system", "content":
                "You are a helpful assistant. Answer using ONLY the provided "
                "movie plots. If the answer isn't in the plots, say so."},
            {"role": "user", "content":
                f"Movie plots:\n{context}\n\nQuestion: {question}"},
        ],
    )
    return resp.choices[0].message.content

# Generate one answer per question
candidates = []
for q in QUESTIONS:
    ans = answer_naively(q["question"], MOVIES)
    candidates.append({"question_id": q["id"], "answer": ans})
    print(f"── {q['id']} ({q['difficulty']}) ──")
    print(f"   Q: {q['question']}")
    print(f"   A: {ans}\n")

── q1 (easy) ──
   Q: What is the name of the rebel leader who recruits Neo?
   A: The name of the rebel leader who recruits Neo is Morpheus.

── q2 (easy) ──
   Q: Who is Andy's favorite toy at the start of Toy Story?
   A: Andy's favorite toy at the start of Toy Story is Woody the cowboy doll.

── q3 (medium) ──
   Q: In Inception, what happens if you get trapped in limbo?
   A: The provided movie plots do not specify what happens if you get trapped in limbo in Inception.

── q4 (medium) ──
   Q: Compare how the conflict starts in The Godfather versus Jurassic Park.
   A: In "The Godfather," the conflict starts when rival families propose entering the narcotics trade, and Don Vito Corleone refuses, leading to a war. In "Jurassic Park," the conflict begins when the security systems fail during a storm, allowing the dinosaurs to escape their enclosures.

── q5 (edge (out-of-scope)) ──
   Q: What year does the Star Wars trilogy take place in?
   A: The provided movie plots do not mentio

---

## Section 3 — Why `assert answer == expected` fails

The deck told you this. Now let's SEE it. Run the same question three times
with temperature=1.0 (max variation) and observe.

In [5]:
same_question = "What is the name of the rebel leader who recruits Neo?"
expected_answer = "Morpheus"

print(f"Q: {same_question}")
print(f"Expected answer: {expected_answer!r}\n")

answers = []
for i in range(3):
    a = answer_naively(same_question, MOVIES, temperature=1.0)
    answers.append(a)
    print(f"  Attempt {i+1}: {a!r}")

print("\n═ The naive test ═")
for i, a in enumerate(answers, 1):
    equal = a == expected_answer
    print(f"  Attempt {i} == 'Morpheus'?  {equal}")

print("\nAll three answers are CORRECT. All three FAIL `assert answer == expected`.")
print("This is why we need eval — not testing.")

Q: What is the name of the rebel leader who recruits Neo?
Expected answer: 'Morpheus'

  Attempt 1: 'The rebel leader who recruits Neo is Morpheus.'
  Attempt 2: 'The name of the rebel leader who recruits Neo is Morpheus.'
  Attempt 3: 'The name of the rebel leader who recruits Neo is Morpheus.'

═ The naive test ═
  Attempt 1 == 'Morpheus'?  False
  Attempt 2 == 'Morpheus'?  False
  Attempt 3 == 'Morpheus'?  False

All three answers are CORRECT. All three FAIL `assert answer == expected`.
This is why we need eval — not testing.


**Discussion:**
- All three attempts got the right answer
- All three failed strict equality
- What DID work: someone reading each answer could say "yes, that mentions Morpheus and answers the question"
- That's what LLM-as-judge automates — reading + judging

The mental shift: **don't compare strings, evaluate properties.**

---

## Section 4 — Build the LLM-as-judge (rubric-based)

This is the biggest hands-on block of Day 1. We build a working LLM-as-judge
with structured output (Pydantic — ties back to W2).

**The rubric** — 3 dimensions, each scored 1-4:
- **Accuracy** — is the answer factually correct given the corpus?
- **Groundedness** — does it stay within the provided context (no invention)?
- **Format** — is it well-structured and appropriately concise?

In [6]:
# Cell 4a — Define the rubric as text (this goes into the judge prompt)

RUBRIC = """Evaluate the answer on three dimensions. For each, give a score 1-4:

ACCURACY (1-4)
  4 = fully correct given the corpus
  3 = mostly correct, minor issues
  2 = partially correct, significant issues
  1 = incorrect or misleading

GROUNDEDNESS (1-4)
  4 = every claim traces to the corpus
  3 = mostly grounded, small unsupported additions
  2 = mix of grounded and invented content
  1 = substantially invented or hallucinated

FORMAT (1-4)
  4 = clear, appropriately concise, well-structured
  3 = clear but slightly verbose or terse
  2 = confusing structure or wrong length
  1 = badly formatted, hard to read"""

print(RUBRIC)

Evaluate the answer on three dimensions. For each, give a score 1-4:

ACCURACY (1-4)
  4 = fully correct given the corpus
  3 = mostly correct, minor issues
  2 = partially correct, significant issues
  1 = incorrect or misleading

GROUNDEDNESS (1-4)
  4 = every claim traces to the corpus
  3 = mostly grounded, small unsupported additions
  2 = mix of grounded and invented content
  1 = substantially invented or hallucinated

FORMAT (1-4)
  4 = clear, appropriately concise, well-structured
  3 = clear but slightly verbose or terse
  2 = confusing structure or wrong length
  1 = badly formatted, hard to read


In [7]:
# Cell 4b — Define the structured output schema (Pydantic, W2 pattern)

class JudgeVerdict(BaseModel):
    accuracy: int      = Field(ge=1, le=4)
    groundedness: int  = Field(ge=1, le=4)
    format_score: int  = Field(ge=1, le=4)
    reasoning: str     = Field(min_length=20, max_length=500)

print("Schema fields:")
for name, field in JudgeVerdict.model_fields.items():
    print(f"  {name:15s}  {field.annotation.__name__}")

Schema fields:
  accuracy         int
  groundedness     int
  format_score     int
  reasoning        str


In [8]:
# Cell 4c — The judge function

def judge(question: str, answer: str, corpus: list, ideal: str = "") -> JudgeVerdict:
    """LLM-as-judge with a rubric. Returns structured verdict."""
    context = "\n\n".join(f"=== {m['title']} ===\n{m['plot']}" for m in corpus)
    ideal_hint = f"\n\nIdeal answer (for reference): {ideal}" if ideal else ""
    
    resp = client.chat.completions.parse(
        model=JUDGE_MODEL,
        temperature=0.0,
        response_format=JudgeVerdict,
        messages=[
            {"role": "system", "content":
                f"You are a strict evaluator of LLM answers.\n\n{RUBRIC}\n\n"
                "Return your scores and a brief reasoning explaining the scores."},
            {"role": "user", "content":
                f"Corpus:\n{context}\n\n"
                f"Question: {question}{ideal_hint}\n\n"
                f"Candidate answer to evaluate: {answer}"},
        ],
    )
    return resp.choices[0].message.parsed

# Test the judge on one candidate
q = QUESTIONS[0]
c = candidates[0]
verdict = judge(q["question"], c["answer"], MOVIES, ideal=q["ideal"])

print(f"Q: {q['question']}")
print(f"A: {c['answer']}\n")
print(f"  Accuracy:     {verdict.accuracy}/4")
print(f"  Groundedness: {verdict.groundedness}/4")
print(f"  Format:       {verdict.format_score}/4")
print(f"  Reasoning:    {verdict.reasoning}")

Q: What is the name of the rebel leader who recruits Neo?
A: The name of the rebel leader who recruits Neo is Morpheus.

  Accuracy:     4/4
  Groundedness: 4/4
  Format:       4/4
  Reasoning:    The candidate answer is fully accurate as it correctly identifies Morpheus as the rebel leader who recruits Neo, which is directly supported by the corpus. The answer is also well-grounded, as it directly references the information provided in the corpus about 'The Matrix'. The format is clear, concise, and well-structured, providing a straightforward response to the question.


In [9]:
# Cell 4d — Run the judge on ALL 6 candidates

verdicts = []
for q, c in zip(QUESTIONS, candidates):
    v = judge(q["question"], c["answer"], MOVIES, ideal=q["ideal"])
    verdicts.append({"q_id": q["id"], "difficulty": q["difficulty"], "verdict": v, "answer": c["answer"]})
    print(f"  {q['id']} ({q['difficulty']:22s}) acc={v.accuracy} ground={v.groundedness} fmt={v.format_score}")

  q1 (easy                  ) acc=4 ground=4 fmt=4
  q2 (easy                  ) acc=4 ground=4 fmt=4
  q3 (medium                ) acc=4 ground=4 fmt=4
  q4 (medium                ) acc=4 ground=4 fmt=4
  q5 (edge (out-of-scope)   ) acc=4 ground=4 fmt=4
  q6 (inference             ) acc=3 ground=3 fmt=4


In [10]:
# Cell 4e — Results table

print(f"  {'Q':4s} {'Difficulty':22s} {'Acc':>4s} {'Grd':>4s} {'Fmt':>4s}  Reasoning (first 60 chars)")
print(f"  {'---':4s} {'----------':22s} {'---':>4s} {'---':>4s} {'---':>4s}  {'-'*40}")
for row in verdicts:
    v = row["verdict"]
    print(f"  {row['q_id']:4s} {row['difficulty']:22s} "
          f"{v.accuracy:>4d} {v.groundedness:>4d} {v.format_score:>4d}  "
          f"{v.reasoning[:60]}...")

  Q    Difficulty              Acc  Grd  Fmt  Reasoning (first 60 chars)
  ---  ----------              ---  ---  ---  ----------------------------------------
  q1   easy                      4    4    4  The candidate answer is fully accurate as it correctly ident...
  q2   easy                      4    4    4  The candidate answer is fully accurate as it correctly ident...
  q3   medium                    4    4    4  The candidate answer accurately states that the provided mov...
  q4   medium                    4    4    4  The candidate answer accurately describes the initiation of ...
  q5   edge (out-of-scope)       4    4    4  The candidate answer accurately states that the Star Wars tr...
  q6   inference                 3    3    4  The candidate answer correctly identifies "Titanic" as invol...


**Discussion moment:**
- Look at the reasoning field for each row. Does the judge's rationale match what YOU'D say if you were grading?
- Which question got the lowest score? Why?
- What did q5 (out-of-scope Star Wars) get? Did the naive answer refuse, or invent?
- What did q6 (inference — 'which involve romance?') get?

**What you just built** is the same pattern you'll port to `src/eval/judge.py`
in Day 2's capstone lab. The Pydantic schema, the rubric-in-system-prompt, the
structured output — all of it transfers directly.

---

## Section 5 — Pairwise comparison

Rubric scoring gives you numbers. Pairwise gives you a cleaner signal: 
'which of these two answers is better?' Less biased than absolute scoring.

We'll write two variants of the answer generator (different system prompts)
and pit them against each other.

In [11]:
# Cell 5a — Two answer variants

SYSTEM_A_BRIEF = (
    "Answer using only the provided plots. Be brief — one sentence when possible."
)

SYSTEM_B_THOROUGH = (
    "Answer using only the provided plots. Provide context and explain your "
    "reasoning. Be thorough but stay grounded."
)

def answer_with_prompt(question: str, corpus: list, system_prompt: str) -> str:
    context = "\n\n".join(f"=== {m['title']} ===\n{m['plot']}" for m in corpus)
    resp = client.chat.completions.create(
        model=CANDIDATE_MODEL,
        temperature=0.0,
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user",   "content": f"Plots:\n{context}\n\nQuestion: {question}"},
        ],
    )
    return resp.choices[0].message.content

# Generate both variants for all questions
print("Generating variant A (brief)...")
answers_a = [answer_with_prompt(q["question"], MOVIES, SYSTEM_A_BRIEF) for q in QUESTIONS]
print("Generating variant B (thorough)...")
answers_b = [answer_with_prompt(q["question"], MOVIES, SYSTEM_B_THOROUGH) for q in QUESTIONS]

# Show one comparison
print(f"\nExample — {QUESTIONS[3]['question']}\n")
print(f"A (brief):    {answers_a[3]}\n")
print(f"B (thorough): {answers_b[3]}")

Generating variant A (brief)...
Generating variant B (thorough)...

Example — Compare how the conflict starts in The Godfather versus Jurassic Park.

A (brief):    In The Godfather, the conflict begins with Don Vito Corleone's refusal to enter the narcotics trade, leading to a war with rival families, while in Jurassic Park, the conflict arises from the failure of security systems during a storm, allowing dinosaurs to escape and threaten the characters.

B (thorough): In "The Godfather," the conflict begins with a refusal by Don Vito Corleone to engage in the narcotics trade, which leads to tensions with rival families. This refusal sets off a chain reaction of violence and power struggles, as the rival families see an opportunity to challenge Corleone's authority and expand their own influence. The assassination attempt on Don Vito serves as a catalyst that pulls his son Michael into the family business, escalating the conflict further.

In contrast, "Jurassic Park" presents a conflic

In [12]:
# Cell 5b — Pairwise judge

class PairwiseVerdict(BaseModel):
    winner: str        = Field(pattern="^[AB]$")  # 'A' or 'B'
    reasoning: str     = Field(min_length=20, max_length=400)

def pairwise_judge(question: str, answer_a: str, answer_b: str, corpus: list) -> PairwiseVerdict:
    """Show the judge two answers side-by-side. Ask which is better."""
    context = "\n\n".join(f"=== {m['title']} ===\n{m['plot']}" for m in corpus)
    resp = client.chat.completions.parse(
        model=JUDGE_MODEL,
        temperature=0.0,
        response_format=PairwiseVerdict,
        messages=[
            {"role": "system", "content":
                "You are comparing two candidate answers to the same question. "
                "Judge on: accuracy, groundedness, and appropriate length/format. "
                "Pick A or B as the winner and briefly explain why."},
            {"role": "user", "content":
                f"Corpus:\n{context}\n\n"
                f"Question: {question}\n\n"
                f"Answer A: {answer_a}\n\n"
                f"Answer B: {answer_b}"},
        ],
    )
    return resp.choices[0].message.parsed

# Run pairwise on all 6
results = []
for q, a, b in zip(QUESTIONS, answers_a, answers_b):
    v = pairwise_judge(q["question"], a, b, MOVIES)
    results.append({"q_id": q["id"], "winner": v.winner, "reasoning": v.reasoning})
    print(f"  {q['id']}  winner={v.winner}   {v.reasoning[:70]}...")

wins_a = sum(1 for r in results if r["winner"] == "A")
wins_b = sum(1 for r in results if r["winner"] == "B")
print(f"\nTally: A (brief) won {wins_a}   |   B (thorough) won {wins_b}")

  q1  winner=A   Both answers correctly identify Morpheus as the rebel leader who recru...
  q2  winner=A   Answer A is concise and directly answers the question by stating that ...
  q3  winner=B   Answer B provides a more comprehensive and detailed explanation of wha...
  q4  winner=B   Answer B provides a more detailed and nuanced comparison of the confli...
  q5  winner=B   Answer B is more comprehensive and explicitly states that the provided...
  q6  winner=B   Answer B is the winner because it accurately identifies the movies fro...

Tally: A (brief) won 2   |   B (thorough) won 4


**Discussion:**
- Which prompt variant won overall?
- On which questions? (E.g., did 'thorough' win on the multi-fact q4?)
- Would you have picked the same winners?
- **But wait** — did A win because A really is better, or because A was always shown first? That's Section 6.

---

## Section 6 — Position bias, live

LLM judges tend to prefer the first option shown. This is **position bias** —
the deck named it, now we see it happen.

**Experiment:** run the same pairwise 3 times with the order flipped between
runs. If the judge flips its winner just because the order changed, that's
position bias caught in the act.

In [13]:
# Cell 6a — Take one question, run pairwise 3 times with different orderings

test_q  = QUESTIONS[3]["question"]  # multi-fact question
ans_a   = answers_a[3]              # brief variant
ans_b   = answers_b[3]              # thorough variant

print(f"Q: {test_q}\n")
print("Running 3 rounds. Watch if the judge flips.\n")

# Round 1: A first, B second
r1 = pairwise_judge(test_q, ans_a, ans_b, MOVIES)
print(f"  Round 1  (A shown first, B second):  winner={r1.winner}")
print(f"           {r1.reasoning[:80]}...\n")

# Round 2: B first, A second (flipped)
r2 = pairwise_judge(test_q, ans_b, ans_a, MOVIES)
print(f"  Round 2  (B shown first, A second):  winner={r2.winner}")
print(f"           NOTE: 'winner' is A or B by position, not by content.")
print(f"           So 'A' in Round 2 refers to the THOROUGH answer (originally B).\n")

# Round 3: A first again
r3 = pairwise_judge(test_q, ans_a, ans_b, MOVIES)
print(f"  Round 3  (A first again):            winner={r3.winner}")

print("\nDid the winner flip between rounds 1 and 2?")
print("If yes: position bias caught in the act.")
print("If no: this judge is more robust — but bias still exists on other questions.")

Q: Compare how the conflict starts in The Godfather versus Jurassic Park.

Running 3 rounds. Watch if the judge flips.

  Round 1  (A shown first, B second):  winner=B
           Answer B provides a more detailed and nuanced comparison of the conflicts in "Th...

  Round 2  (B shown first, A second):  winner=A
           NOTE: 'winner' is A or B by position, not by content.
           So 'A' in Round 2 refers to the THOROUGH answer (originally B).

  Round 3  (A first again):            winner=B

Did the winner flip between rounds 1 and 2?
If yes: position bias caught in the act.
If no: this judge is more robust — but bias still exists on other questions.


In [14]:
# Cell 6b — Position-flip mitigation
# Randomise which answer is shown first, then remap the result back.

def pairwise_judge_debiased(question: str, answer_variant_a: str,
                            answer_variant_b: str, corpus: list,
                            rng: random.Random | None = None) -> dict:
    """Flip position randomly. Return winner in ORIGINAL A/B labels (not shown order)."""
    rng = rng or random.Random()
    flip = rng.random() < 0.5
    
    if flip:
        # Show B first, A second
        v = pairwise_judge(question, answer_variant_b, answer_variant_a, corpus)
        # 'A' in the response means 'the one shown first' → which is variant B
        actual_winner = "B" if v.winner == "A" else "A"
    else:
        # Show A first (normal)
        v = pairwise_judge(question, answer_variant_a, answer_variant_b, corpus)
        actual_winner = v.winner
    
    return {"winner": actual_winner, "flipped": flip, "reasoning": v.reasoning}

# Run 5 rounds with debiasing — random flip each time
rng = random.Random(42)  # seeded for reproducibility

print(f"Q: {test_q}\n")
print("5 debiased rounds (random position-flip each time):\n")

for i in range(5):
    r = pairwise_judge_debiased(test_q, ans_a, ans_b, MOVIES, rng=rng)
    order = "B first" if r["flipped"] else "A first"
    print(f"  Round {i+1}  ({order})  actual_winner={r['winner']}")

print("\nWith random flipping, the winner should reflect content, not position.")

Q: Compare how the conflict starts in The Godfather versus Jurassic Park.

5 debiased rounds (random position-flip each time):

  Round 1  (A first)  actual_winner=B
  Round 2  (B first)  actual_winner=B
  Round 3  (B first)  actual_winner=B
  Round 4  (B first)  actual_winner=B
  Round 5  (A first)  actual_winner=B

With random flipping, the winner should reflect content, not position.


**Discussion moment:**
- Section 5 gave us a tally. Was it reliable?
- The mitigation in 6b is what real systems use. In your capstone lab tomorrow,
  apply position-flip when you run pairwise on YOUR two prompts.
- Other judge biases exist too (length bias — judges prefer longer answers;
  self-bias — judges prefer their own model). We're just doing position
  today; the deck's instructor notes cover the others.

---

## Section 7 — Critic-Creator loop

The final eval pattern. Two LLMs (or two prompts) alternate:
- **Creator** drafts an answer
- **Critic** critiques it using the rubric
- **Creator** revises based on the critique
- Repeat 2-3 rounds

Watch the answer improve round-by-round.

In [15]:
# Cell 7a — The Creator prompt

def creator(question: str, corpus: list, previous_critique: str = "") -> str:
    """Draft (or revise) an answer. If a critique is provided, address it."""
    context = "\n\n".join(f"=== {m['title']} ===\n{m['plot']}" for m in corpus)
    
    if previous_critique:
        user_msg = (
            f"Corpus:\n{context}\n\n"
            f"Question: {question}\n\n"
            f"Previous critique to address:\n{previous_critique}\n\n"
            f"Write an improved answer that addresses the critique."
        )
    else:
        user_msg = (
            f"Corpus:\n{context}\n\n"
            f"Question: {question}\n\n"
            f"Write a clear, grounded answer."
        )
    
    resp = client.chat.completions.create(
        model=CANDIDATE_MODEL,
        temperature=0.0,
        messages=[
            {"role": "system", "content":
                "You are the Creator. Draft answers using only the provided corpus."},
            {"role": "user", "content": user_msg},
        ],
    )
    return resp.choices[0].message.content

In [16]:
# Cell 7b — The Critic prompt

def critic(question: str, draft: str, corpus: list) -> str:
    """Critique a draft answer using the rubric."""
    context = "\n\n".join(f"=== {m['title']} ===\n{m['plot']}" for m in corpus)
    
    resp = client.chat.completions.create(
        model=JUDGE_MODEL,
        temperature=0.0,
        messages=[
            {"role": "system", "content":
                f"You are the Critic. Read a draft answer and critique it using "
                f"this rubric:\n\n{RUBRIC}\n\n"
                "Point out specific weaknesses. Be constructive — say WHAT to "
                "change and HOW. If the draft is already strong, say so."},
            {"role": "user", "content":
                f"Corpus:\n{context}\n\n"
                f"Question: {question}\n\n"
                f"Draft answer to critique:\n{draft}"},
        ],
    )
    return resp.choices[0].message.content

In [17]:
# Cell 7c — The loop

# Pick a harder question (the multi-fact comparison one)
hard_q = QUESTIONS[3]["question"]

print(f"Q: {hard_q}\n")
print("═" * 70)

# Round 1: creator drafts, critic critiques
draft_1 = creator(hard_q, MOVIES)
print(f"\n── ROUND 1 · Creator draft ──\n{draft_1}")

critique_1 = critic(hard_q, draft_1, MOVIES)
print(f"\n── ROUND 1 · Critic critique ──\n{critique_1}")

# Round 2: creator revises based on critique
draft_2 = creator(hard_q, MOVIES, previous_critique=critique_1)
print(f"\n── ROUND 2 · Creator revision ──\n{draft_2}")

critique_2 = critic(hard_q, draft_2, MOVIES)
print(f"\n── ROUND 2 · Critic critique ──\n{critique_2}")

# Round 3: creator revises again
draft_3 = creator(hard_q, MOVIES, previous_critique=critique_2)
print(f"\n── ROUND 3 · Creator final ──\n{draft_3}")

Q: Compare how the conflict starts in The Godfather versus Jurassic Park.

══════════════════════════════════════════════════════════════════════

── ROUND 1 · Creator draft ──
In "The Godfather," the conflict begins with Don Vito Corleone's refusal to enter the narcotics trade, which provokes rival families and leads to a violent war. This refusal sets off a chain of events that draws his son Michael into the family's criminal activities after an assassination attempt on Don Vito.

In contrast, the conflict in "Jurassic Park" arises from the failure of security systems during a storm, which allows the cloned dinosaurs to escape their enclosures. This breakdown of control creates immediate danger for the characters on the island, as they must confront the consequences of the park's creation and the inherent risks of reviving extinct species.

While "The Godfather" centers on a deliberate choice that ignites a power struggle, "Jurassic Park" features an external crisis that results from

**Discussion moment:**
- Did the answer materially improve from Round 1 to Round 3?
- What specific weaknesses did the critic catch that the creator addressed?
- Did any critique feel wrong or nit-picky?
- When would this loop stop being useful? (Usually 2-3 rounds; more rarely helps.)

**Where you'll use this pattern:**
- In tomorrow's capstone lab (Step 5) on one of YOUR harder golden questions
- In production: some systems run Critic-Creator on ambiguous or high-stakes
  answers before returning them to users. Expensive but effective.

---

## Section 8 — Wrap: the 4 patterns you'll use tomorrow

In 90 minutes you built:

| Pattern | Section | What it gives you | Cost |
|---|---|---|---|
| **Rubric-based LLM-as-judge** | 4 | Scores on 3 dimensions with reasoning | 1 judge call per (q, a) |
| **Pairwise comparison** | 5 | 'Which is better?' cleaner signal than absolute scoring | 1 judge call per (q, a_A, a_B) |
| **Position-bias mitigation** | 6 | Random-flip to remove ordering bias | Same cost, cleaner result |
| **Critic-Creator loop** | 7 | Systematic prompt improvement | 2-3 rounds × 2 calls per round |

**Tomorrow's capstone lab** (Day 2, in-class + take-home for the milestone):

1. **Stakeholder Map** for YOUR capstone — new material
2. **Build 20-question golden set** for YOUR corpus — new material
3. **Port `judge` to `src/eval/judge.py`** and run on YOUR golden set — Section 4 applied
4. **Port `pairwise_judge_debiased` to `src/eval/pairwise.py`** and run between YOUR two prompts — Section 5 + 6 applied
5. **Run Critic-Creator on ONE hard golden question** — Section 7 applied
6. **Finalise ADR v1 (Locked) + DR #1 1-pager** — M1 close-out

The notebook code isn't the final answer — tomorrow you'll integrate these
into `src/eval/*` modules with SQLite persistence, prompt versioning, and
proper error handling. Same shape, capstone-integrated.

**That is your M1 deliverable.**